<div style="text-align: center;">

# EXAMPLE EXERCISE - RISK SCORES AND FINANCIAL METRICS

</div>

This notebook demonstrates an end-to-end physical climate risk workflow for a sample real estate portfolio. It starts from asset data, submits the portfolio to the Physrisk API, retrieves asset-level risk scores, groups assets into hazard-specific clusters, and then requests financial metrics that translate physical hazard exposure into monetary loss measures.

## Notebook Setup

The first cells load the libraries used throughout the notebook. `pandas` handles tabular portfolio and response data, `requests` communicates with the Physrisk API, `plotly` creates the visualizations, and `shapely` converts the cluster geometries returned by the API into map-ready polygons.

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv
from shapely import MultiPolygon, Polygon, wkt

from plots import plot_exceedance_curves, plot_var, plot_es

The API configuration is read from `../.env` so that credentials and service URLs are not hard-coded in the notebook. The resulting `headers` object sends the configured API key with each request and requests JSON responses.

In [2]:
_ = load_dotenv("../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")
headers = {"X-API-Key": API_KEY, "Accept": "application/json"}

---

## 1. Loading And Registering The Asset Portfolio

The workflow begins with a CSV portfolio of real estate assets. Each asset includes an identifier, asset class, building type, coordinates, country, and monetary value.

In [3]:
assets_df = pd.read_csv("../resources/example_portfolio.csv", dtype={"asset_id": str})

Displaying `assets_df` is a quick validation step before sending data to the API. At this point we check that the portfolio has the expected schema, that latitude and longitude are present, and that each asset has a value that can later be used in financial loss calculations.

In [4]:
assets_df

,id,asset_id,name,asset_class,type,location,latitude,longitude,country,value
0,0,1,Ourense 1,RealEstateAsset,Buildings/Industrial,Europe,42.345298,-7.893056,ES,1773000.0
1,1,2,Ourense 2,RealEstateAsset,Buildings/Industrial,Europe,42.304359,-7.846886,ES,1862000.0
2,2,3,Palencia 1,RealEstateAsset,Buildings/Industrial,Europe,42.017684,-4.555006,ES,2059000.0
3,3,4,Venta de Baños 1,RealEstateAsset,Buildings/Industrial,Europe,41.920601,-4.483257,ES,1188000.0
4,4,5,Venta de Baños 2,RealEstateAsset,Buildings/Industrial,Europe,41.914530,-4.491720,ES,2475000.0
5,5,6,Cubillas de Cerrato 1,RealEstateAsset,Buildings/Industrial,Europe,41.802881,-4.474943,ES,3665000.0
6,6,7,Roquetas de Mar 1,RealEstateAsset,Buildings/Industrial,Europe,36.756014,-2.622427,ES,262000.0
7,7,8,Roquetas de Mar 2,RealEstateAsset,Buildings/Industrial,Europe,36.783739,-2.622377,ES,2686000.0
8,8,9,Roquetas de Mar 3,RealEstateAsset,Buildings/Industrial,Europe,36.712404,-2.639418,ES,2297000.0
9,9,10,Almería 1,RealEstateAsset,Buildings/Industrial,Europe,36.855174,-2.418615,ES,2489000.0


The following code displays a preview of the portfolio’s geographic location and financial exposure.

In [5]:
def show_portfolio(assets_df: pd.DataFrame = assets_df) -> go.Figure:
    latitudes: list = assets_df["latitude"].to_list()
    longitudes: list = assets_df["longitude"].to_list()
    
    customdata = np.stack(
        [
            assets_df["asset_id"],
            assets_df["name"],
            assets_df["value"],
        ],
        axis=-1,
    )

    marker_sizes = np.interp(
        assets_df["value"],
        (assets_df["value"].min(), assets_df["value"].max()),
        (8, 30),
    )
    
    center_lat = np.mean(latitudes)
    center_lon = np.mean(longitudes)

    fig = go.Figure()

    fig.add_trace(
        go.Scattermap(
                lat=latitudes,
                lon=longitudes,
                mode="markers",
                marker=dict(
                    symbol="circle",
                    size=marker_sizes,
                    color="rgba(0, 146, 160, 1.0)",
                ),
                customdata=customdata,
                hovertemplate=(
                    "ID: %{customdata[0]}<br>"
                    "Name: %{customdata[1]}<br>"
                    "Value: %{customdata[2]:,.0f} EUR<br>"
                    "<extra></extra>"
                ),
            )
    )

    fig.update_layout(
        map=dict(
            style="carto-positron",
            zoom=5,
            center=dict(
                lat=center_lat,
                lon=center_lon,
            ),
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        width=1000,
        height=600,
        showlegend=False,
    )

    return fig

In [6]:
show_portfolio(assets_df)

The portfolio is then submitted to the Physrisk pipeline. The response returns a `portfolio_code`, which becomes the key used by the rest of the notebook to retrieve risk scores, clusters, exceedance curves, and financial metrics for the same registered portfolio.

The following cluster calculation endpoint is also triggered immediately after registration so that hazard-specific clusters are available for later sections.

> WARNING: This request may take a few minutes to be processed.

In [7]:
request = {
    "username": "client",
    "portfolio_assets": assets_df.to_dict("records"),
    "portfolio_name": "example",
}

portfolio_risks = requests.post(
    url=f"{API_BASE}/physical_asset_pipeline/calculate_portfolio_risks", json=request, headers=headers
)

dict_response = portfolio_risks.json()
portfolio_code = dict_response["inserted_portfolio_code"]

portfolio_clusters = requests.get(
    url=f"{API_BASE}/physical_asset_pipeline/calculate_portfolio_clusters/{portfolio_code}", headers=headers
)

---

## 2. Obtaining Asset-Level Risk Scores

With the portfolio registered, the notebook requests asset-level risk results. These results combine physical hazard information with climate scenario and time-horizon dimensions, producing risk scores for each asset, hazard, indicator, scenario, and forecast horizon available from the API.

In [8]:
request = {
    "portfolio_code": portfolio_code,
}
asset_level_risk = requests.post(
    url=f"{API_BASE}/api/asset_level_risk", json=request, headers=headers
)
dict_asset_level_risk = asset_level_risk.json()

Two small helper functions make the raw API response easier to inspect:

- `available_hazards` lists the hazard and indicator combinations present in the response.
- `risk_scores_from_hazards` filters the response to one hazard-indicator pair and returns a tidy table with asset, scenario, time horizon, and risk score.

This separation is useful because the API response is nested, while the analysis is easier to follow in tabular form.

In [9]:
def available_hazards(asset_level_risk: dict[str, list[dict]]) -> pd.DataFrame:
    """Returns a table with all available hazards-indicators combinations for risk score metrics (acute + chronic)."""
    info_list: list[dict] = asset_level_risk["asset_risks"]

    hazards = set()
    for data in info_list:
        hazard_data = {"hazard": data["hazard"], "indicator": data["indicator"]}
        hazards.add(frozenset(hazard_data.items()))

    df = pd.DataFrame(
        [dict(h) for h in hazards]
        )[["hazard", "indicator"]].sort_values(
            by="hazard"
        ).reset_index(
            drop=True
    )
    
    return df

def risk_scores_from_hazards(asset_level_risk: dict[str, list[dict]], *, hazard: str, indicator: str) -> pd.DataFrame:
    info_list: list[dict] = asset_level_risk["asset_risks"]

    rows = []
    for data in info_list:
        if data["hazard"] != hazard:
            continue
        if data["indicator"] != indicator:
            continue
        
        row = {
            "asset_id": data["client_asset_id"],
            "scenario": data["scenario_name"],
            "time_horizon": data["forecast_horizon"],
            "risk_score": data["risk_score"]
        }

        rows.append(row)

    df = pd.DataFrame(rows).sort_values(by=["scenario", "time_horizon", "asset_id"]).reset_index(drop=True)

    return df

This table is the inventory of hazard-indicator combinations available for risk score analysis. It shows which physical risks can be queried from the current portfolio response, including acute hazards such as `Fire` or `RiverineInundation` and chronic hazards such as `ChronicHeat` or `WaterRisk`.

In [10]:
available_hazards(dict_asset_level_risk)

,hazard,indicator
0,ChronicHeat,mean_degree_days/above/index
1,CoastalInundation,flood_depth
2,Drought,cdd
3,Drought,spi6
4,Fire,fire_probability
5,Landslide,landslide_susceptability
6,RiverineInundation,flood_depth
7,Subsidence,subsidence_susceptability
8,WaterRisk,water_stress
9,Wind,wind_speed/3s


The example below selects `Drought` with the `cdd` indicator and compares risk scores across assets, scenarios, and time horizons. The intended reading is directional: higher scores indicate greater relative risk for the selected hazard and indicator under the corresponding climate scenario and period.

In [11]:
hazard = "Drought"
indicator = "cdd"

risk_scores_from_hazards(dict_asset_level_risk, hazard=hazard, indicator=indicator)

,asset_id,scenario,time_horizon,risk_score
0,0,Historical,1971 - 2010,2
1,1,Historical,1971 - 2010,2
2,2,Historical,1971 - 2010,2
3,3,Historical,1971 - 2010,2
4,4,Historical,1971 - 2010,2
...,...,...,...,...
79,7,RCP 8.5,2035 - 2065,3
80,8,RCP 8.5,2035 - 2065,3
81,9,RCP 8.5,2035 - 2065,3
82,10,RCP 8.5,2035 - 2065,3


---

## 3. Clustering

The clustering section retrieves the hazard clusters associated with the registered portfolio. Clusters aggregate assets that share exposure to the same `acute hazard` footprint, which is especially useful for financial metrics because losses may be correlated within the same spatial hazard event.

In [16]:
clusters = requests.get(
    url=f"{API_BASE}/api/cluster/{portfolio_code}", headers=headers
)

dict_clusters = clusters.json()

try:
    _ = dict_clusters["clusters"]
except KeyError:
    raise KeyboardInterrupt("The clustering response is probably not ready yet. Try again in a few seconds.") from None

In [19]:
portfolio_code

826

These helper functions prepare cluster results for review and visualization:

- `available_clustering_hazards` identifies which hazards have cluster data.
- `get_clusters_by_hazard` joins cluster definitions with the assets assigned to each cluster.
- `plot_clusters_by_hazard` converts the returned WKT geometries into polygons and overlays the clustered assets on a map.

The clustering API currently returns acute hazard clusters.

In [20]:
clusters_asset2cluster = requests.get(
    url=f"{API_BASE}/api/cluster/asset_cluster_info/{portfolio_code}/RiverineInundation", headers=headers
)
asset2cluster_info: list[dict] = clusters_asset2cluster.json()["asset_cluster_info"]

In [18]:
clusters_asset2cluster.json()

{'detail': 'Forbidden: You do not have access to this portfolio'}

In [21]:
def available_clustering_hazards(clusters: dict[str, list[dict]]) -> pd.DataFrame:
    """Returns a table with all available hazard combinations for clustering (only acute)."""
    info_list: list[dict] = clusters["clusters"]

    hazards = set()
    for data in info_list:
        hazards.add(data["hazard"])

    df = pd.DataFrame(
        [{"hazard": h} for h in hazards]
        ).sort_values(
            by="hazard"
        ).reset_index(
            drop=True
    )
    
    return df

def get_clusters_by_hazard(clusters: dict[str, list[dict]], *, hazard: str) -> pd.DataFrame:
    info_list: list[dict] = clusters["clusters"]

    clusters_asset2cluster = requests.get(
        url=f"{API_BASE}/api/cluster/asset_cluster_info/{portfolio_code}/{hazard}", headers=headers
    )
    asset2cluster_info: list[dict] = clusters_asset2cluster.json()["asset_cluster_info"]

    rows = []
    for data in info_list:
        if data["hazard"] != hazard:
            continue

        cluster_id: int = data["cluster_id"]
        cluster_name: str = data["cluster_name"]
        value: float = data["value"]
        geometry: Polygon | MultiPolygon = data["geometry"]

        assets = set()
        for a2c in asset2cluster_info:
            if a2c["cluster_name"] != cluster_name:
                continue

            assets.add(a2c["client_asset_id"])
        
        row = {
            "cluster_id": cluster_id,
            "assets": list(assets),
            "cluster_name": cluster_name,
            "value": value,
            "geometry": geometry,
        }

        rows.append(row)

    return pd.DataFrame(rows)

def plot_clusters_by_hazard(
    clusters: dict[str, list[dict]], 
    *, 
    hazard: str, 
    assets_df: pd.DataFrame = assets_df,
) -> go.Figure:
    fillcolor = "rgba(0, 146, 160, 0.5)"
    linecolor = "rgba(0, 146, 160, 1.0)"

    clusters_df: pd.DataFrame = get_clusters_by_hazard(clusters, hazard=hazard)

    cluster_names: list[str] = clusters_df["cluster_name"].to_list()
    geometries: list[Polygon | MultiPolygon] = clusters_df["geometry"].to_list()
    values: list[float] = clusters_df["value"].to_list()
    assets: list[list[int]] = clusters_df["assets"].to_list()

    fig = go.Figure()

    for n, g, v, a in zip(cluster_names, geometries, values, assets, strict=True):
        g = wkt.loads(g)
        if isinstance(g, Polygon):
            geom = [g]
        elif isinstance(g, MultiPolygon):
            geom = list(g.geoms)
        else:
            raise ValueError(f"Not valid geometry {type(g)}")
        
        for poly in geom:
            lon, lat = poly.exterior.xy

            fig.add_trace(
                go.Scattermap(
                    lon=list(lon),
                    lat=list(lat),
                    mode="lines",
                    fill="toself",
                    fillcolor=fillcolor,
                    line=dict(color=linecolor, width=2),
                    hovertext=f"Name: {n}. Value: {v}",
                    hoverinfo="text",
                )
            )
            
        asset_points: pd.DataFrame = assets_df[assets_df["id"].isin(a)]
        asset_lat: list = asset_points["latitude"].to_list()
        asset_lon: list = asset_points["longitude"].to_list()
        asset_name: list = asset_points["name"].to_list()

        fig.add_trace(
            go.Scattermap(
                lat=asset_lat,
                lon=asset_lon,
                mode="markers",
                marker=dict(
                    symbol="circle",
                    size=10,
                    color=linecolor,
                ),
                text=asset_name
            )
        )

    fig.update_layout(
        map=dict(
            style="carto-positron",
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        width=1000,
        height=600,
        showlegend=False,
    )

    return fig

The hazard list confirms which cluster analyses can be performed for the current portfolio.

In [22]:
available_clustering_hazards(dict_clusters)

,hazard
0,CoastalInundation
1,Combined
2,Fire
3,RiverineInundation
4,Wind


The cluster table links each cluster to the assets contained in it, the cluster value, and the geometry describing the affected footprint. The `value` field is important because it represents the exposed asset value that will later feed the loss and financial metric calculations.

In [23]:
get_clusters_by_hazard(dict_clusters, hazard="Fire")

,cluster_id,assets,cluster_name,value,geometry
0,1,"[3, 4]",Wildfire cluster 1,3663000.0,"MULTIPOLYGON(((-4.477 41.93116666666667,-4.477..."
1,2,[2],Wildfire cluster 2,2059000.0,MULTIPOLYGON(((-4.581833333333334 42.029833333...
2,3,[8],Wildfire cluster 3,2297000.0,"MULTIPOLYGON(((-2.6395 36.712583333333335,-2.6..."
3,4,[7],Wildfire cluster 4,2686000.0,MULTIPOLYGON(((-2.622416666666667 36.783916666...
4,5,[1],Wildfire cluster 5,1862000.0,"MULTIPOLYGON(((-7.834 42.32083333333333,-7.834..."
5,6,"[0, 5]",Wildfire cluster 6,5438000.0,"MULTIPOLYGON(((-7.89475 42.3495,-7.89475 42.34..."


The map provides a spatial check of the cluster output. Polygon outlines show the hazard footprints returned by the API, while asset markers show which portfolio locations fall inside each cluster. This helps verify that the financial metrics are being computed on geographically coherent groups of exposed assets.

In [24]:
fig = plot_clusters_by_hazard(dict_clusters, hazard="Fire")
fig.show()

---

# 4. Financial Metrics

## 4.1. Cluster-Level Metrics

Cluster-level financial metrics focus on one hazard and one cluster. The selected `hazard` and `cluster_id` define the slice of the portfolio for which the notebook will request exceedance curves and loss metrics.

Check `available_clustering_hazards` and `get_clusters_by_hazard` to search for request data availability.

In [25]:
hazard = "RiverineInundation"
cluster_id = 2090605040

The cluster exceedance curve endpoint returns loss amounts associated with different exceedance probabilities. The cluster metric endpoint returns summary financial risk measures for the same hazard and cluster. Together, they describe both the loss distribution and selected point estimates derived from it.

In [26]:
clusters_exceedance_curve = requests.get(
    url=f"{API_BASE}/api/cluster/exceedance_curves/{portfolio_code}/{cluster_id}/{hazard}", headers=headers
)

dict_clusters_exceedance_curve = clusters_exceedance_curve.json()

clusters_metrics = requests.get(
    url=f"{API_BASE}/api/cluster/metric/{portfolio_code}/{cluster_id}/{hazard}", headers=headers
)

dict_clusters_metrics = clusters_metrics.json()

The exceedance curve plots monetary loss against exceedance probability. Moving toward lower probabilities corresponds to rarer but more severe events. Comparing scenario lines helps show how the distribution of potential losses changes under different climate pathways.

In [27]:
plot_exceedance_curves(dict_clusters_exceedance_curve, name=f"Cluster {cluster_id}; {hazard}", selected_term="short", dark=True)

Value at Risk (VaR) summarizes a high-percentile loss level for the selected term and scenario. It answers the question: what loss threshold is only exceeded with a specified low probability?

In [28]:
plot_var(dict_clusters_metrics, name=f"Cluster {cluster_id}; {hazard}", selected_term="short", dark=True)

Expected Shortfall (ES) complements VaR by focusing on the tail beyond the VaR threshold. It estimates the average loss conditional on being in those severe outcomes, so it is often more informative for stress and tail-risk analysis.

In [29]:
plot_es(dict_clusters_metrics, name=f"Cluster {cluster_id}; {hazard}", selected_term="short", dark=True)

## 4.2. Portfolio-Level Metrics

The portfolio-level view repeats the same financial risk logic, but aggregates across all assets in the registered portfolio for the selected hazard. This is the relevant level for understanding total exposure rather than the contribution of a single cluster.

In [30]:
hazard = "Combined"

The portfolio endpoints return exceedance curves and financial metrics after aggregating hazard losses across the whole registered portfolio. These results capture how individual asset and cluster losses combine into total portfolio risk.

In [31]:
portfolio_hazard_exceedance = requests.get(
    url=f"{API_BASE}/physical_asset_pipeline/exceedance_curves/{portfolio_code}/{hazard}", headers=headers
)

dict_portfolio_hazard_exceedance = portfolio_hazard_exceedance.json()

portfolio_metrics = requests.get(
    url=f"{API_BASE}/physical_asset_pipeline/metrics/{portfolio_code}/{hazard}", headers=headers
)

dict_portfolio_metrics = portfolio_metrics.json()

The portfolio exceedance curve shows the distribution of total portfolio losses for the selected hazard. Compared with the cluster-level curve, this view can include multiple exposed assets and therefore reflects concentration and diversification effects across the registered portfolio.

In [32]:
plot_exceedance_curves(dict_portfolio_hazard_exceedance, name=f"Portfolio; {hazard}", selected_term="short", dark=True)

The portfolio VaR plot summarizes severe-but-plausible total losses at the portfolio level for the selected term and scenario set. This is useful for comparing portfolio-wide downside risk across climate pathways.

In [33]:
plot_var(dict_portfolio_metrics, name=f"Portfolio; {hazard}", selected_term="short", dark=True)

The final plot shows portfolio Expected Shortfall. It emphasizes the average severity of losses beyond the VaR threshold and provides a tail-risk view of the portfolio's wildfire exposure.

In [34]:
plot_es(dict_portfolio_metrics, name=f"Portfolio; {hazard}", selected_term="short", dark=True)